# 20장 · 접수일에 적용되는 판본을 골라 인용하기

장비실에서 “신청은 며칠 전까지인가요?”라는 문의를 받았습니다. 구판에는 3일, 개정판에는 5일이라고 쓰여 있습니다. 검색이 두 숫자를 모두 찾은 것과 해당 학생에게 맞는 하나를 고른 일은 다릅니다. 접수일 하나만 바꾸어 답이 어떻게 바뀌는지 확인합니다.

## 실습 준비

전체 실습 묶음을 풀고 노트북을 엽니다. 같은 폴더에서 시작해 자료 묶음의 `pre-mid/documents`를 읽습니다. 공용 준비 파일에 뒤에서 사용할 함수도 있으므로 **아래 셀에서 가져오는 이름**부터 살펴보세요. 처음 한 번 커널에서 `%pip install pypdf langchain-openai langchain-core chromadb python-dotenv`를 실행하고 커널을 재시작하세요. 작업 폴더의 .env에 OPENAI_API_KEY를 둡니다. 첫 색인 생성과 새 질문 임베딩은 API를 호출합니다.

In [1]:
from pathlib import Path
import sys

# 압축을 푼 폴더 또는 그 아래에서 실습 자료의 위치를 찾습니다.
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pre-mid/documents/rag/rag_lab.py").is_file()), None)
if root is None:
    raise FileNotFoundError("전체 실습 묶음을 풀고 pre-mid 폴더 옆에서 실행하세요.")
rag_dir = root / "pre-mid/documents/rag"
sys.path.insert(0, str(rag_dir))

## 시행일과 사용자 날짜를 구분하기

자료 폴더의 loan-spring.pdf, loan-summer.pdf를 펼치세요. 적용 기준은 **접수일**입니다. 3월 31일에 접수해 4월에 장비를 받더라도 적용되는 안내는 3월판입니다. 표에는 ‘접수일, 수령일, 적용 판본, 신규 신청 마감, 근거 쪽’을 채웁니다. 접수일이 없는 질문에서 어떤 값을 먼저 물어볼지도 적습니다.

In [2]:
from pypdf import PdfReader
# 판본을 고르기 전, 두 파일에 실제로 무엇이 쓰여 있는지 먼저 읽습니다.
for name in ["loan-spring.pdf", "loan-summer.pdf"]:
    p = PdfReader(rag_dir / name).pages[0]
    print(name, p.extract_text().strip())

loan-spring.pdf 실습용 문서  |  물결 장비실
신청과 승인
장비 대여 안내  1.0판  |  2027년 3월 1일부터 3월 31일까지
이 판은 2027년 3월 1일부터 3월 31일까지 접수된 신규 신청에 적용합니다.
장비를 받으려는 날짜보다 3일 전까지 신청서를 제출하세요.
승인 여부는 담당자가 접수 다음 근무일에 신청자에게 알립니다.
신청 기준일은 사용일이 아니라 접수일입니다.
교육용 가상 기관    물결 장비실     1.0판 / 1쪽
loan-summer.pdf 실습용 문서  |  물결 장비실
신청과 승인
장비 대여 안내  1.1판  |  2027년 4월 1일부터
이 판은 2027년 4월 1일부터 접수된 신규 신청에 적용합니다.
장비를 받으려는 날짜보다 5일 전까지 신청서를 제출하세요.
승인 여부는 담당자가 접수 다음 근무일에 신청자에게 알립니다.
신청 기준일은 사용일이 아니라 접수일입니다.
교육용 가상 기관    물결 장비실     1.1판 / 1쪽


검색 키가 같고 문서명도 같아 보이면 파일명이나 최신 날짜만 보고 개정판을 고르기 쉽습니다. 이 자료에는 명시된 적용 구간이 있습니다. 기계가 비교할 수 있게 날짜를 적고, 범위 안에 드는 판본만 후보로 보냅니다. 학생에게 적용되는 판본은 담당자가 원문을 읽어 등록한 정보입니다.

In [3]:
from datetime import date
from rag_lab import answer_from_documents

editions = [
    ("loan-spring.pdf", date(2027,3,1), date(2027,3,31)),
    ("loan-summer.pdf", date(2027,4,1), date.max)]
def active_pages(received):
    rows = []
    for filename, first, last in editions:
        if first <= received <= last:
            for page_no, page in enumerate(PdfReader(rag_dir / filename).pages, 1):
                rows.append({"id": f"{filename}#p{page_no}", "source": filename,
                             "page": page_no, "text": page.extract_text()})
    return rows

received = date(2027,3,31)
q = "3월 31일 접수하는 신규 신청입니다. 수령 며칠 전까지 제출하나요?"
candidates = active_pages(received)
print([d["id"] for d in candidates])
print(answer_from_documents(q, candidates))

['loan-spring.pdf#p1', 'loan-spring.pdf#p2']
3월 31일 접수하는 신규 신청은 2027년 3월 1일부터 3월 31일까지 접수된 신규 신청에 적용됩니다. 장비를 받으려는 날짜보다 3일 전까지 신청서를 제출해야 합니다.[loan-spring.pdf#p1]


## 인용 검토에서 숫자보다 먼저 확인할 것

파일명을 열어 1쪽의 적용 기간을 읽고, 이어 숫자가 나온 문장이 ‘신규 신청’인지 확인합니다. 승인된 신청의 시간 변경은 2쪽 예외가 적용됩니다. 접수일을 정확히 골라도 업무의 종류가 다르면 답이 틀릴 수 있습니다.

## 적용 판본 자체가 없는 날짜

자료에는 3월 이전 접수의 규정이 없습니다. 사용자가 2월에 접수했다고 하면 ‘가장 오래된 판본’을 쓰면 될까요? 날짜를 모델에 넘기기 전에 적용 가능한 자료 수를 셉니다. 빈 후보를 별도 경로로 처리하면 자료가 없는 날짜에서 비용을 들여 답을 추측할 이유가 없습니다.

In [4]:
past = active_pages(date(2027, 2, 28))
if not past:
    print("2027년 2월 28일 접수에 적용되는 안내를 보유하지 않습니다.")

2027년 2월 28일 접수에 적용되는 안내를 보유하지 않습니다.


이 사례는 3월판과 4월판 중 무엇이 더 비슷한지로 해결되지 않습니다. 적용 범위 밖이라는 사실이 이미 정해져 있기 때문입니다. 접수일을 모르는 상황은 또 다릅니다. 날짜를 먼저 물으면 자료가 있는지 결정할 수 있으므로 추가 질문을 선택합니다.

날짜 필터의 결과가 한 판본으로 좁혀져도 여러 쪽이 남습니다. 먼저 적용 자격을 판단하고, 그다음에 필요한 조항을 고르는 순서입니다. “최신판만 남겨 두기”는 지난달 접수 기록을 검토하는 업무에서는 답할 자료 자체를 없앨 수 있습니다.

## 실습 · 경계 날짜와 예외를 교차해 문의하기

3월 31일과 4월 1일의 신규 신청을 비교합니다. 이어서 “이미 승인됐고 수령 시간만 변경”으로 조건을 바꿉니다. 두 문서의 어느 쪽이 필요한지 답을 받기 전에 적고, 응답이 인용한 쪽의 실제 문장을 확인합니다.

In [5]:
# 날짜와 질문의 표현을 짝으로 바꿔 보세요.

<details class="notebook-answer"><summary>모범 풀이와 판정 기준</summary><pre><code>for day, text in [(date(2027,3,31),&quot;신규 신청은 수령 며칠 전까지 하나요?&quot;),
                  (date(2027,4,1),&quot;신규 신청은 수령 며칠 전까지 하나요?&quot;),
                  (date(2027,4,1),&quot;승인된 신청의 수령 시간만 바꾸면 언제까지 알려야 하나요?&quot;)]:
    print(day, answer_from_documents(text, active_pages(day)))</code></pre><p>신규 신청은 3월판 1쪽에서 3일, 4월판 1쪽에서 5일입니다. 승인된 신청의 시간 변경은 4월판 2쪽에서 수령 전날 오후 4시까지입니다. 답에 숫자만 맞고 판본·쪽이 다르면 근거 없는 인용으로 기록합니다.</p><p>집필 시 실행 결과</p><pre>2027-03-31 신규 신청은 장비를 받으려는 날짜보다 3일 전까지 신청서를 제출해야 합니다.[loan-spring.pdf#p1]
2027-04-01 신규 신청은 장비를 받으려는 날짜보다 5일 전까지 해야 합니다.[loan-summer.pdf#p1]
2027-04-01 승인된 신청의 수령 시간만 바꾸는 경우에는 수령 전날 오후 4시까지 담당자에게 새 시간을 알려야 합니다.[loan-summer.pdf#p2]</pre></details>

## 독립 실습 · 신청 종류가 달라지면 무엇이 바뀌는가

3월 안내(loan-spring.pdf)와 4월 안내(loan-summer.pdf)를 엽니다. 서로 이름이 비슷한 아래 세 사람에게 보내는 답변을 각각 씁니다. 두 사람이 같은 판본을 사용해도 인용 쪽수가 달라질 수 있습니다.

| 문의 | 답변에 필요한 조건 | 선택한 파일·쪽 | 회신 |
|---|---|---|---|
| 3월 31일 접수할 신규 신청자입니다. 수령일은 4월입니다. | | | |
| 4월 1일 신규 신청을 합니다. | | | |
| 4월에 이미 승인을 받았습니다. 수령 시간과 장비 종류를 둘 다 바꿉니다. | | | |

<details class="notebook-answer"><summary>판정 예시와 해설</summary><p>3월 31일 접수 신규 신청은 3월판 1쪽에 따라 수령 3일 전까지, 4월 1일 접수 신규 신청은 4월판 1쪽에 따라 수령 5일 전까지입니다. 수령 시간과 장비 종류를 모두 바꿀 때는 시간만 바꾸는 예외에 해당하지 않습니다. 4월판 2쪽 마지막 문장을 인용해 신규 신청 절차를 안내하고, 신규 신청 마감까지 알려 주려면 같은 판 1쪽도 인용합니다.</p></details>

## 다음 수업에 가져갈 것

세 질문의 접수일·업무 유형·답·인용을 보관하세요. 다음 장에서는 올바른 판본을 골라도 안내서에 해당 절차 자체가 없는 질문을 처리합니다.

서울 개인 거래일을 6월 30일에서 7월 1일로 바꾸세요. 지점과 개인·가맹점 구분은 유지한 채 두 질문의 유효 기간과 정책 페이지를 찾습니다. [36쪽 운영 편람 연속 실습](pre-mid/documents/transfer/handbook-lab.md)